[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rgarzonj/python_data_science_basics/blob/main/collab/reading_the_data.ipynb)

> **Google Colab note:** all required packages (`numpy`, `pandas`, `matplotlib`, `seaborn`, `ipywidgets`) are pre-installed in Colab — just run the cells in order.

# Reading the Data: An Executive Tour of EDA

**An actuary in Paris just sent you a CSV file with 678,000 motor insurance policies. Before any modelling, before any pricing — what's actually in there?**

---

This notebook is a hands-on tour of **Exploratory Data Analysis** (EDA): the detective work that begins every serious data science project. The data is real — it's `freMTPL2freq`, the canonical French Motor Third-Party Liability dataset used by actuaries worldwide. We've taken a 30,000-row sample to keep things runnable.

### What you'll do
1. Open the file and form first impressions
2. Find the dominant pattern that defines the problem
3. Hunt for data-quality issues — the things real datasets always hide
4. Profile each variable, one at a time
5. Look for what predicts a claim
6. Avoid the most common rookie mistake (the *exposure trap*)
7. Ask: where would an analyst go from here?

> **Why this matters for an executive.** EDA is where most of the value of a data science project is created or destroyed. A model trained on dirty, mis-understood data is worse than no model. By the end of this notebook, you'll know what good EDA looks like — and what to ask your team.

## Setup

Run the cell below to load the toolkit and the dataset. The CSV is embedded in the notebook itself (no internet needed).

In [ ]:
import base64, gzip, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Visual style
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.titlesize": 13,
                     "axes.titleweight": "bold", "font.family": "serif",
                     "axes.spines.top": False, "axes.spines.right": False})

# Embedded data — a 30,000-row stratified sample of freMTPL2freq.
# The full dataset has 677,991 policies. Source: dutangc/CASdatasets (R package).
_CSV_B64 = "..."

raw = base64.b64decode(_CSV_B64)
with gzip.open(io.BytesIO(raw), 'rt') as f:
    df = pd.read_csv(f)

print(f"✅ Loaded {len(df):,} policies, {df.shape[1]} columns")
print(f"   Source: freMTPL2freq (French Motor TPL, ~30k stratified sample)")


## Stage 1 · First glance

Three lines you run on every dataset, ever. They tell you:
- **`.shape`** — how many rows × columns?
- **`.head()`** — what does a row look like?
- **`.dtypes`** — what type is each column?

Together: in 5 seconds you know roughly what you're dealing with.

In [ ]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("First 5 rows:")
display(df.head())
print("\nColumn types:")
print(df.dtypes)

### What do these columns mean?

The actuary's email said: *"Standard French TPL pricing variables. Should be self-explanatory."* It's not. Click any column below to reveal what it actually means.

In [ ]:
column_glossary = {
    "IDpol":      ("Policy ID", "A unique identifier per policy. Not predictive — used for joins."),
    "ClaimNb":    ("Number of claims", "How many third-party-liability claims this policy generated. THE TARGET VARIABLE."),
    "Exposure":   ("Exposure (in years)", "Fraction of a year the policy was in force. A policy active for 6 months has Exposure=0.5. CRITICAL — see Stage 6."),
    "VehPower":   ("Vehicle power", "An integer rating of engine power (4-15). Higher = more powerful car = generally riskier."),
    "VehAge":     ("Vehicle age", "Age of the car in years."),
    "DrivAge":    ("Driver age", "Age of the policyholder in years."),
    "BonusMalus": ("Bonus-Malus level", "France's no-claims discount system. Starts at 100, drops 5%/year claim-free (down to 50), rises after claims (up to 350). Lower = better record."),
    "VehBrand":   ("Vehicle brand", "Anonymised brand label (B1, B2, ..., B14). 11 values. The actual brand mapping was stripped."),
    "VehGas":     ("Fuel type", "'Regular' (petrol) or 'Diesel'."),
    "Area":       ("Area code", "A density-based area code (A=rural, F=urban). 6 values."),
    "Density":    ("Population density", "Inhabitants per km² where the driver lives."),
    "Region":     ("Region", "French administrative region (22 values, pre-2016 borders)."),
}

dropdown = widgets.Dropdown(options=list(column_glossary.keys()),
                             description="Column:",
                             style={"description_width": "100px"})
out = widgets.Output()

def on_change(change):
    if change["type"] != "change" or change["name"] != "value":
        return
    with out:
        clear_output()
        col = change["new"]
        title, desc = column_glossary[col]
        display(Markdown(f"**{col}** — {title}\n\n{desc}"))
        print(f"\nIn this dataset:")
        if df[col].dtype.kind in "biufc":
            print(f"  range:  {df[col].min()} → {df[col].max()}")
            print(f"  median: {df[col].median()}")
            print(f"  unique: {df[col].nunique():,}")
        else:
            print(f"  unique: {df[col].nunique():,}")
            print(f"  top 5:  {dict(df[col].value_counts().head().to_dict())}")

dropdown.observe(on_change)
display(dropdown, out)

## Stage 2 · The dominant pattern

Now we look at the *target variable* — the thing we'd ultimately want to predict. In this dataset that's `ClaimNb`, the number of claims a policy generated.

**Before you run the cell:** what fraction of motor policies do you think generate at least one claim per year? Hold a number in your head.

In [ ]:
claim_rate = (df["ClaimNb"] > 0).mean() * 100
zero_claims = (df["ClaimNb"] == 0).sum()
total       = len(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left — the binary view: any claim or not?
counts = df["ClaimNb"].apply(lambda x: "≥1 claim" if x > 0 else "0 claims").value_counts()
axes[0].bar(counts.index, counts.values, color=["#5d6e3a", "#a73e2c"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,}\n({v/total*100:.1f}%)", ha="center", fontweight="bold")
axes[0].set_title("Most policies never claim")
axes[0].set_ylabel("Number of policies")

# Right — the full distribution
counts_full = df["ClaimNb"].value_counts().sort_index()
axes[1].bar(counts_full.index.astype(int), counts_full.values, color="#5d6e3a")
axes[1].set_yscale("log")
axes[1].set_title("Claim count distribution (log scale)")
axes[1].set_xlabel("Number of claims in year")
axes[1].set_ylabel("Number of policies (log)")
for i, v in zip(counts_full.index, counts_full.values):
    if v > 0:
        axes[1].text(i, v * 1.3, f"{int(v):,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n{claim_rate:.1f}% of policies generated at least one claim.")
print(f"That means {100-claim_rate:.1f}% of customers paid premium and never claimed.")

> 💡 **The defining fact of insurance EDA.** About **3.7% of policies** generate any claim in a year. The other 96% pay premium and cost the insurer almost nothing. This *extreme imbalance* shapes everything: how you sample, how you measure model accuracy, how you price.
> 
> A model that predicts *"no claim"* for everyone is **96% accurate**. That's why "accuracy" is a useless metric in insurance — the model would be useless. Instead, actuaries care about *expected claim frequency* and *expected severity*, multiplied to get *expected loss*.

## Stage 3 · Hunting for data-quality issues

Real datasets always hide problems. Sentinel values disguised as data, impossible ranges, suspicious clustering at round numbers. **Before you trust any dataset, you go looking for trouble.**

Run the cell below. It scans each numeric column for common red flags.

In [ ]:
def scan_column(col):
    """Look for data-quality red flags in a numeric column."""
    s = df[col]
    flags = []

    # Negative values where they should not exist
    if col in ["Exposure", "VehAge", "DrivAge", "Density", "VehPower", "BonusMalus", "ClaimNb"]:
        if (s < 0).any():
            flags.append(f"❌ {(s<0).sum()} negative values (impossible)")

    # Impossible ranges
    if col == "Exposure" and (s > 1).any():
        flags.append(f"⚠️  {(s>1).sum()} policies with Exposure > 1 year (impossible for annual contracts)")
    if col == "DrivAge":
        if (s < 18).any(): flags.append(f"⚠️  {(s<18).sum()} drivers under 18 (illegal in France)")
        if (s > 100).any(): flags.append(f"⚠️  {(s>100).sum()} drivers over 100")
        if (s > 90).any():  flags.append(f"⚠️  {(s>90).sum()} drivers over 90 (suspicious — likely placeholder)")
    if col == "VehAge":
        if (s > 50).any(): flags.append(f"⚠️  {(s>50).sum()} vehicles over 50 years old (suspicious)")
        if (s > 90).any(): flags.append(f"❌ {(s>90).sum()} vehicles over 90 — these are placeholders for missing values")

    # Suspicious clustering at sentinel values (100, 99, 999, etc.)
    for sentinel in [99, 100, 999]:
        n = (s == sentinel).sum()
        if n > len(df) * 0.001 and col in ["VehAge", "DrivAge"]:
            flags.append(f"⚠️  {n} rows have exactly {sentinel} (often a placeholder for missing)")

    if not flags:
        return [f"✅ {col} — no obvious issues"]
    return [f"📋 {col}:"] + ["    " + f for f in flags]

print("Scanning numeric columns for data quality red flags:\n")
for col in ["Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density", "ClaimNb"]:
    for line in scan_column(col):
        print(line)

> 💡 **What real data looks like.** This is the canonical actuarial dataset, used in dozens of academic papers — and *it still has placeholders, impossible values, and outliers*. Your team's data is no different. **An EDA that doesn't surface issues like these isn't done.**
>
> Notice especially the **Exposure > 1** and **VehAge = 100** issues. The first is impossible (a 1-year contract can't be in force for 1.5 years); the second is a placeholder where the true age was missing. Either silently corrupts a model trained on this data.

### How would you handle these?

Run the cell below to see three common strategies, applied to the `Exposure > 1` issue:

In [ ]:
bad = df[df["Exposure"] > 1]
print(f"Policies with Exposure > 1: {len(bad)} ({len(bad)/len(df)*100:.2f}%)")
print(f"Their Exposure values: min={bad['Exposure'].min():.2f}, max={bad['Exposure'].max():.2f}")
print()

# Strategy 1 — drop them
df_drop = df[df["Exposure"] <= 1]
print(f"Strategy 1 (drop): {len(df_drop):,} rows kept. Loses {len(bad)} rows.")

# Strategy 2 — clip to 1
df_clip = df.copy()
df_clip["Exposure"] = df_clip["Exposure"].clip(upper=1.0)
print(f"Strategy 2 (clip to 1): {len(df_clip):,} rows kept, all with Exposure ≤ 1.")

# Strategy 3 — flag and keep
df_flag = df.copy()
df_flag["exposure_capped"] = df_flag["Exposure"] > 1
df_flag["Exposure"] = df_flag["Exposure"].clip(upper=1.0)
print(f"Strategy 3 (flag): {len(df_flag):,} rows kept, with a new column flagging the {df_flag['exposure_capped'].sum()} affected ones.")
print()
print("Strategy 3 is usually best — it preserves information for downstream analysis")
print("while capping the impossible values. Always flag, rarely drop, never silently clean.")

## Stage 4 · Profile each variable, one at a time

Now we look at each column in detail. The shape of a single variable tells you a lot:
- Is it normally distributed, skewed, or bimodal?
- Are there obvious outliers?
- Are categorical values balanced or dominated by a few?

**Pick a column from the dropdown** and the right plot type will be chosen automatically.

In [ ]:
numeric_cols = ["Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
categorical_cols = ["VehBrand", "VehGas", "Area", "Region"]
all_cols = numeric_cols + categorical_cols

def profile_column(col):
    s = df[col]
    if col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        # Histogram
        axes[0].hist(s, bins=40, color="#5d6e3a", alpha=0.85)
        axes[0].axvline(s.mean(),   color="#a73e2c", linestyle="--", linewidth=1.5, label=f"mean={s.mean():.1f}")
        axes[0].axvline(s.median(), color="#1a4a6b", linestyle="--", linewidth=1.5, label=f"median={s.median():.1f}")
        axes[0].set_title(f"{col} — distribution"); axes[0].legend()
        # Boxplot to expose outliers
        axes[1].boxplot(s.dropna(), vert=False, widths=0.6)
        axes[1].set_title(f"{col} — box plot (outliers visible)")
        axes[1].set_yticks([])
        plt.tight_layout(); plt.show()
        # Stats
        print(f"\n{col} stats:")
        print(f"  mean: {s.mean():>10.2f}")
        print(f"  std:  {s.std():>10.2f}")
        print(f"  min:  {s.min():>10.2f}")
        print(f"  25%:  {s.quantile(.25):>10.2f}")
        print(f"  50%:  {s.quantile(.50):>10.2f}")
        print(f"  75%:  {s.quantile(.75):>10.2f}")
        print(f"  max:  {s.max():>10.2f}")
        print(f"  skew: {s.skew():>10.2f}  (>1 = right-skewed)")
    else:
        counts = s.value_counts()
        fig, ax = plt.subplots(figsize=(11, max(3, len(counts)*0.3)))
        ax.barh(counts.index.astype(str), counts.values, color="#5d6e3a")
        ax.invert_yaxis()
        ax.set_title(f"{col} — frequency")
        ax.set_xlabel("Number of policies")
        for i, v in enumerate(counts.values):
            ax.text(v + max(counts.values)*0.01, i, f"{v:,}", va="center", fontsize=9)
        plt.tight_layout(); plt.show()
        print(f"\n{col}: {len(counts)} unique values")
        print(f"   most common: {counts.index[0]} ({counts.iloc[0]:,} = {counts.iloc[0]/len(df)*100:.1f}%)")

widgets.interact(profile_column,
                 col=widgets.Dropdown(options=all_cols, value="DrivAge",
                                      description="Column:",
                                      style={"description_width": "100px"}));

### Things to look for as you click through

- **`Exposure`** — bimodal: a big spike at 1.0 (full-year policies, ~25%) and the rest spread fairly evenly below it. Median is around 0.49 — most policies in the snapshot are partway through the year.
- **`DrivAge`** — right-skewed with a single peak around 35–45. The youngest drivers (under 25) are rare; numbers thin out steadily after 60.
- **`BonusMalus`** — extremely concentrated at the floor: **56% of drivers sit at exactly 50**, the maximum no-claims discount. The rest taper off with a long thin tail up to ~200.
- **`Region`** — Centre dominates (~24% of policies). Real-world data is rarely uniform across geography.
- **`VehBrand`** — three brands (B1, B2, B12) account for ~72%. The other 8 are a long tail.

> 💡 **The point:** every column has a *shape*. Knowing the shape is half the work of an analyst. Asymmetric distributions need different statistical treatment than symmetric ones; long-tailed categoricals need pooling or a "rare" bucket.

## Stage 5 · What predicts a claim?

Now we look at *relationships*. For each feature, we ask: **does the claim rate change as this feature changes?** If yes, it's a useful predictor.

We use a different trick for each variable type:
- **Numeric:** bin into deciles and plot claim frequency per bin
- **Categorical:** plot claim frequency per category

In [ ]:
def claim_rate_by_feature(col, n_bins=10):
    df_clean = df[df["Exposure"] > 0].copy()  # avoid divide-by-zero

    if col in numeric_cols:
        # Bin into deciles, compute claim frequency PER UNIT EXPOSURE
        df_clean["_bin"] = pd.qcut(df_clean[col], q=n_bins, duplicates="drop")
        agg = df_clean.groupby("_bin", observed=True).apply(
            lambda g: pd.Series({
                "claim_rate": g["ClaimNb"].sum() / g["Exposure"].sum(),
                "n_policies": len(g),
                "midpoint": g[col].median()
            }),
            include_groups=False,
        ).reset_index()
        fig, ax = plt.subplots(figsize=(10, 4.5))
        ax.plot(agg["midpoint"], agg["claim_rate"], marker="o",
                linewidth=2, color="#a73e2c", markersize=7)
        ax.set_xlabel(col); ax.set_ylabel("Claim frequency (claims per policy-year)")
        ax.set_title(f"Claim frequency vs {col}")
        avg_rate = df_clean["ClaimNb"].sum() / df_clean["Exposure"].sum()
        ax.axhline(avg_rate, color="#888", linestyle="--", alpha=0.7,
                   label=f"book average ({avg_rate:.3f})")
        ax.legend()
        plt.tight_layout(); plt.show()
    else:
        agg = df_clean.groupby(col, observed=True).apply(
            lambda g: pd.Series({
                "claim_rate": g["ClaimNb"].sum() / g["Exposure"].sum(),
                "n_policies": len(g),
            }),
            include_groups=False,
        ).reset_index().sort_values("claim_rate", ascending=True)
        fig, ax = plt.subplots(figsize=(11, max(3, len(agg)*0.32)))
        ax.barh(agg[col].astype(str), agg["claim_rate"], color="#a73e2c")
        avg_rate = df_clean["ClaimNb"].sum() / df_clean["Exposure"].sum()
        ax.axvline(avg_rate, color="#888", linestyle="--", linewidth=1.5,
                   label=f"book average ({avg_rate:.3f})")
        ax.set_xlabel("Claim frequency (claims per policy-year)")
        ax.set_title(f"Claim frequency by {col}")
        ax.legend()
        plt.tight_layout(); plt.show()

widgets.interact(claim_rate_by_feature,
                 col=widgets.Dropdown(options=all_cols, value="DrivAge",
                                      description="Feature:",
                                      style={"description_width": "100px"}),
                 n_bins=widgets.IntSlider(value=10, min=4, max=20, step=1,
                                          description="Bins (numeric):"));

### What you'll see as you click through

- **`DrivAge`** — young drivers (under 25) have *much* higher claim frequency (~0.16, more than double the book average of 0.074). After 25 the curve is roughly flat through the working ages, then drifts down. **Not the U-shape you might expect** — the very-old uptick that's common in some datasets is not pronounced here.
- **`BonusMalus`** — strong upward trend. The system *works*: drivers at higher B-M values genuinely claim more.
- **`Area`** — flat across rural levels (A, B, C all around 0.064), then climbs steadily to F (urban, ~0.10). It's a step up at level D rather than a smooth gradient.
- **`VehAge`** — fairly flat (around 0.07–0.08) across most ages. The "older cars are driven less" effect is subtler in this dataset than in textbooks.
- **`VehGas`** — Diesel slightly higher (0.076) than Regular (0.072). Small but real.

> 💡 **What this is.** This is *univariate signal detection* — finding which variables *individually* predict the target. It's the first analytical step in pricing. Your team will then build models that combine these signals while controlling for their interactions.

## Stage 6 · The exposure trap (the most common rookie mistake)

This is where 90% of new actuaries trip over their own feet.

A naive analyst would compute: *"What's the average number of claims by region?"* and plot it. Watch what happens when we forget about `Exposure`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — wrong way: average ClaimNb directly
wrong = df.groupby("Region")["ClaimNb"].mean().sort_values()
axes[0].barh(wrong.index, wrong.values, color="#a73e2c")
axes[0].set_title("WRONG — mean(ClaimNb) by region\n(ignores how long each policy was in force)")
axes[0].set_xlabel("Avg ClaimNb per policy")

# Right — right way: total claims / total exposure
right = df.groupby("Region").apply(
    lambda g: g["ClaimNb"].sum() / g["Exposure"].sum(),
    include_groups=False,
).sort_values()
axes[1].barh(right.index, right.values, color="#5d6e3a")
axes[1].set_title("RIGHT — claims per policy-YEAR\n(normalised by exposure)")
axes[1].set_xlabel("Claim frequency (claims / policy-year)")

plt.tight_layout(); plt.show()

# Show the difference
diff = pd.DataFrame({"wrong": wrong, "right": right})
diff["pct_diff"] = (diff["right"] - diff["wrong"]) / diff["wrong"] * 100
diff = diff.sort_values("pct_diff", ascending=False)
print("\nTop regions where the two methods most disagree:")
print(diff.head(5).round(3).to_string())

> 💡 **Why this matters.** The two charts can rank regions in *different orders*. A region with many short-duration policies looks low-risk under the wrong method, even if its underlying claim frequency is high.
>
> **The exposure normalisation is non-negotiable.** Anytime you look at counts of rare events (claims, defaults, conversions, deaths), you must normalise by the time-at-risk that produced them. Forgetting this is the most common mistake in junior actuarial work — and the second-most-common mistake in business analytics generally.

## Stage 7 · Geography

Insurance is geographic at heart — risk is correlated with where people live and drive. Let's look at the regional pattern, properly normalised.

In [ ]:
region_stats = df.groupby("Region").apply(
    lambda g: pd.Series({
        "policies":    len(g),
        "exposure":    g["Exposure"].sum(),
        "claim_freq":  g["ClaimNb"].sum() / g["Exposure"].sum(),
        "avg_density": g["Density"].mean(),
    }),
    include_groups=False,
).sort_values("claim_freq", ascending=True)

fig, ax = plt.subplots(figsize=(11, 8))

# Color regions by sample size — gray out the unreliable ones
small = region_stats["policies"] < 500
colors = ["#cccccc" if s else "#a73e2c" for s in small]
ax.barh(region_stats.index, region_stats["claim_freq"], color=colors)

avg = df["ClaimNb"].sum() / df["Exposure"].sum()
ax.axvline(avg, color="black", linestyle="--", linewidth=1, alpha=0.6,
           label=f"national average ({avg:.3f})")

for i, (region, row) in enumerate(region_stats.iterrows()):
    flag = "  ⚠ small sample" if row["policies"] < 500 else ""
    ax.text(row["claim_freq"] + avg*0.02, i,
            f"  n={int(row['policies']):,}{flag}",
            va="center", fontsize=9,
            color="#999" if row["policies"] < 500 else "#444")

ax.set_xlabel("Claim frequency (claims / policy-year)")
ax.set_title("French motor claim frequency by region\n(red = trustworthy sample, gray = small sample, sorted low → high)")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

### What this tells the actuary

- **Highest claim frequency** (among regions with enough exposure to trust the estimate): **Rhône-Alpes** (0.097) and **Aquitaine** (0.091) sit clearly above the national average of 0.074.
- **Île-de-France** (Paris and surroundings) — *despite the stereotype* — is roughly average at 0.081. The capital region's reputation for high accident rates does not show up cleanly in this dataset.
- **Lowest among well-sampled regions**: Languedoc-Roussillon (0.057), Lorraine (0.058), and Centre (0.060).
- **Watch the sample-size warning.** The very top of the bar chart (Franche-Comté, Picardie, Champagne-Ardenne) shows extreme frequencies — but those regions have only 25–160 policy-years of exposure, so a single claim moves the rate substantially. For regional pricing factors, actuaries always weight by exposure or apply credibility adjustments.

This is exactly the kind of finding that feeds into a *territorial pricing factor* — one of the most regulated parts of insurance pricing — *and* the kind that requires care: a naive ranking would give wildly unstable rates to small regions.

## What you've done

Run the cell below to summarise the dataset — what an analyst would put at the top of a one-page memo to the underwriting team.

In [ ]:
claim_rate_clean = df[df['Exposure'] > 0].apply(
    lambda r: r['ClaimNb'] / r['Exposure'] if r['Exposure'] > 0 else 0, axis=1
)
total_exposure = df['Exposure'].sum()
total_claims = df['ClaimNb'].sum()

print("=" * 65)
print("EDA SUMMARY · freMTPL2freq sample")
print("=" * 65)
print(f"Records:          {len(df):,} policies")
print(f"Total exposure:   {total_exposure:,.1f} policy-years")
print(f"Total claims:     {int(total_claims):,}")
print(f"Claim frequency:  {total_claims/total_exposure:.4f} claims/policy-year")
print(f"Pct with claim:   {(df['ClaimNb']>0).mean()*100:.2f}%")
print()
print("DATA QUALITY ISSUES FOUND:")
print(f"  · {(df['Exposure']>1).sum()} policies with Exposure > 1 year (impossible)")
print(f"  · {(df['DrivAge']>90).sum()} drivers over 90 (likely placeholder)")
print(f"  · {(df['VehAge']>50).sum()} vehicles over 50 years (suspicious)")
print()
print("STRONGEST UNIVARIATE PREDICTORS (ordered by signal strength):")
print(f"  · DrivAge     — young drivers (<25) have ~2x book frequency")
print(f"  · BonusMalus  — strong monotonic relationship; the system works")
print(f"  · Area        — urban (D-F) noticeably higher than rural (A-C)")
print(f"  · Region      — Rhone-Alpes & Aquitaine clearly above average,")
print(f"                  watch small-sample regions for instability")
print()
print("RECOMMENDED NEXT STEPS:")
print("  1. Cap Exposure at 1.0 (with flag column)")
print("  2. Top-code DrivAge at 90 (with flag column)")
print("  3. Convert ClaimNb to a frequency target (claims/exposure)")
print("  4. Build a Poisson GLM with DrivAge, BonusMalus, Area, Region")
print("  5. Use credibility weighting for small regions")
print("  6. Hold out 20% for validation")

## What good EDA looks like — for executives

You've just been through a real exploratory data analysis. A few takeaways to keep:

**1. EDA is a question-driven process, not a checklist.** Each step asked a specific question (*"is the data trustworthy?"*, *"what predicts the target?"*) and the next step depended on the answer.

**2. Real datasets always have problems.** This one is published, peer-reviewed, used in dozens of academic papers — and it still has impossible values, placeholders, and outliers. Yours has more.

**3. The single most important pattern is usually visible in the first 10 minutes.** Here it was the 96/4 split between zero-claim and claiming policies. Find that pattern early; it shapes everything that follows.

**4. Beware the exposure trap.** When you measure rare events (claims, defaults, churn, deaths), always normalise by time-at-risk. Forgetting this is the #1 source of misleading executive dashboards.

**5. Univariate first, then relationships, then models.** Don't let your team jump to modelling before they understand each variable in isolation.

### Questions to ask your team

- *"Show me the distribution of every variable in the dataset."* If they can't, they haven't done EDA.
- *"What data quality issues did you find?"* If "none", they didn't look hard enough.
- *"What's our claim frequency, and how do you define it?"* The answer should mention exposure.
- *"Which features show the strongest univariate signal?"* Tells you what's likely to be most important.

---

*🎉 You've completed an end-to-end EDA on a real-world insurance dataset. The same toolkit, scaled up to your own data, drives most of the analytical groundwork at every modern insurer.*